# Дополнительная часть: Аугментация данных и балансировка классов

Этот блокнот:
1. Загружает основные данные из `dataset`
2. Добавляет изображения из `add_positive` и `add_negative` для балансировки
3. Применяет аугментацию данных
4. Обучает модели и генерирует отчёт с отображением аугментированных версий

In [44]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import base64
from datetime import datetime
from io import BytesIO
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import random

# Фиксируем все seed'ы
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
# ==========================================
# НАСТРОЙКИ
# ==========================================
DATASET_FOLDER = 'dataset'
ADD_POSITIVE_FOLDER = 'add_positive'
ADD_NEGATIVE_FOLDER = 'add_negative'
IMAGE_SIZE = (128, 128)
PIXELS_PER_CELL = (8, 8)
CELLS_PER_BLOCK = (2, 2)
ORIENTATIONS = 9

OUTPUT_CSV = 'results_augmented.csv'
OUTPUT_HTML = 'report_augmented.html'

# ==========================================
# АУГМЕНТАЦИЯ
# ==========================================
def augment_image(img):
    """Применяет случайные аугментации к изображению"""
    img_aug = img.copy()
    h, w = img_aug.shape[:2]
    
    # 1. Случайный поворот (-30 до +30 градусов)
    angle = np.random.uniform(-30, 30)
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    # Используем BORDER_CONSTANT с белым фоном вместо BORDER_REPLICATE
    img_aug = cv2.warpAffine(img_aug, rotation_matrix, (w, h), 
                             borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    
    # 2. Случайное масштабирование (0.8 до 1.2)
    scale = np.random.uniform(0.8, 1.2)
    new_h, new_w = int(h * scale), int(w * scale)
    img_aug = cv2.resize(img_aug, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    # Центрируем после масштабирования
    start_y = max(0, (h - new_h) // 2)
    start_x = max(0, (w - new_w) // 2)
    end_y = min(h, start_y + new_h)
    end_x = min(w, start_x + new_w)
    
    img_resized = np.ones((h, w, 3), dtype=np.uint8) * 255  # Белый фон
    img_resized[start_y:end_y, start_x:end_x] = img_aug[:end_y-start_y, :end_x-start_x]
    img_aug = img_resized
    
    # 3. Случайное изменение яркости (-20 до +20)
    brightness = np.random.randint(-20, 20)
    img_aug = cv2.convertScaleAbs(img_aug, alpha=1.0, beta=brightness)
    
    # 4. Случайное изменение контраста (0.75 до 1.25)
    contrast = np.random.uniform(0.75, 1.25)
    img_aug = cv2.convertScaleAbs(img_aug, alpha=contrast, beta=0)
    
    # 5. Случайное добавление шума (с вероятностью 30%)
    if np.random.random() < 0.3:
        noise = np.random.normal(0, 10, img_aug.shape).astype(np.uint8)
        img_aug = cv2.add(img_aug, noise)
    
    return img_aug
# ==========================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==========================================
def img_to_base64(filepath):
    with open(filepath, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def get_hog_visualization(filepath):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, IMAGE_SIZE)
    fd, hog_image = hog(img_resized,
                        orientations=ORIENTATIONS,
                        pixels_per_cell=PIXELS_PER_CELL,
                        cells_per_block=CELLS_PER_BLOCK,
                        visualize=True,
                        block_norm='L2-Hys')
    hog_image_rescaled = (hog_image / hog_image.max() * 255).astype(np.uint8)
    return fd, hog_image_rescaled

def array_to_base64(arr):
    _, buffer = cv2.imencode('.jpg', arr)
    return base64.b64encode(buffer).decode('utf-8')

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

def compute_hog(filepath):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, IMAGE_SIZE)
    fd = hog(img_resized,
             orientations=ORIENTATIONS,
             pixels_per_cell=PIXELS_PER_CELL,
             cells_per_block=CELLS_PER_BLOCK,
             visualize=False,
             block_norm='L2-Hys')
    return fd

In [45]:
print("=== Шаг 1: Загрузка данных ===")

features = []
labels = []
filenames = []

def load_from_folder(folder, label, prefix=''):
    count = 0
    if not os.path.exists(folder):
        return count
    for filename in os.listdir(folder):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        filepath = os.path.join(folder, filename)
        fd = compute_hog(filepath)
        features.append(fd)
        labels.append(label)
        filenames.append(f"{prefix}{filename}")
        count += 1
    return count

n_pos_main = load_from_folder(DATASET_FOLDER, None, '')  # временно
# Перечитаем с правильными метками
features, labels, filenames = [], [], []

for filename in sorted(os.listdir(DATASET_FOLDER)):
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    label = int(filename.split('_')[-1].split('.')[0])
    filepath = os.path.join(DATASET_FOLDER, filename)
    fd = compute_hog(filepath)
    features.append(fd)
    labels.append(label)
    filenames.append(filename)

print(f"Из dataset: {len(filenames)} изображений")

n_add_pos = load_from_folder(ADD_POSITIVE_FOLDER, 1, 'add_pos_')
n_add_neg = load_from_folder(ADD_NEGATIVE_FOLDER, 0, 'add_neg_')
print(f"Добавлено позитивных: {n_add_pos}, негативных: {n_add_neg}")

X = np.array(features)
y = np.array(labels)

print(f"\nДо аугментации: Позитивных={sum(y==1)}, Негативных={sum(y==0)}")

# СНАЧАЛА split, ПОТОМ аугментация
X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    X, y, filenames, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} (позитивных: {sum(y_train==1)})")
print(f"Test:  {len(X_test)} (позитивных: {sum(y_test==1)})")

=== Шаг 1: Загрузка данных ===
Из dataset: 200 изображений
Добавлено позитивных: 17, негативных: 21

До аугментации: Позитивных=117, Негативных=121
Train: 190 (позитивных: 93)
Test:  48 (позитивных: 24)


In [46]:
print("\n=== Шаг 2: Аугментация ОБУЧАЮЩЕЙ выборки ===")

# Аугментируем ОБА класса для баланса
X_train_aug = list(X_train)
y_train_aug = list(y_train)
files_train_aug = list(files_train)
augmented_imgs_for_report = []  # Для отображения в отчёте

# Аугментируем по 2 версии для каждого изображения в train
for idx in range(len(y_train)):
    orig_name = files_train[idx].replace('add_pos_', '').replace('add_neg_', '')
    filepath = None
    for folder in [DATASET_FOLDER, ADD_POSITIVE_FOLDER, ADD_NEGATIVE_FOLDER]:
        test_path = os.path.join(folder, orig_name)
        if os.path.exists(test_path):
            filepath = test_path
            break
    
    if filepath is None:
        continue
    
    img = cv2.imread(filepath)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 2 аугментированные версии для каждого изображения
    for aug_idx in range(2):
        img_aug = augment_image(img_rgb)
        img_aug_gray = cv2.cvtColor(img_aug, cv2.COLOR_RGB2GRAY)
        img_aug_resized = cv2.resize(img_aug_gray, IMAGE_SIZE)
        
        fd = hog(img_aug_resized,
                 orientations=ORIENTATIONS,
                 pixels_per_cell=PIXELS_PER_CELL,
                 cells_per_block=CELLS_PER_BLOCK,
                 visualize=False,
                 block_norm='L2-Hys')
        
        X_train_aug.append(fd)
        y_train_aug.append(y_train[idx])
        files_train_aug.append(f"aug_{aug_idx}_{files_train[idx]}")
        augmented_imgs_for_report.append(img_aug)

X_train = np.array(X_train_aug)
y_train = np.array(y_train_aug)
files_train = files_train_aug

print(f"После аугментации train: {len(X_train)} изображений")
print(f"Позитивных: {sum(y_train==1)}, Негативных: {sum(y_train==0)}")
print(f"Test остался чистым: {len(X_test)} изображений\n")


=== Шаг 2: Аугментация ОБУЧАЮЩЕЙ выборки ===
После аугментации train: 570 изображений
Позитивных: 279, Негативных: 291
Test остался чистым: 48 изображений



In [47]:
# ==========================================
# ШАГ 3: ОБУЧЕНИЕ МОДЕЛЕЙ
# ==========================================
print("\n=== Шаг 3: Обучение моделей ===")

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "SVM (RBF kernel)": make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
all_metrics = []

for name, model in models.items():
    print(f"Обучение {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Не кукла (0)', 'Кукла (1)'],
                                    output_dict=True, zero_division=0)
    
    results[name] = {'model': model, 'y_pred': y_pred, 'accuracy': acc, 'report': report}
    
    all_metrics.append({
        'Модель': name,
        'Accuracy': f"{acc:.4f}",
        'Precision (Кукла)': f"{report['Кукла (1)']['precision']:.4f}",
        'Recall (Кукла)': f"{report['Кукла (1)']['recall']:.4f}",
        'F1-Score (Кукла)': f"{report['Кукла (1)']['f1-score']:.4f}",
        'Ошибок': int(np.sum(y_pred != y_test))
    })
    
    print(f"Accuracy: {acc:.4f} | Recall (Кукла): {report['Кукла (1)']['recall']:.4f}")
    print("-" * 40)

metrics_df = pd.DataFrame(all_metrics)

best_recall_idx = metrics_df['Recall (Кукла)'].astype(float).idxmax()
best_recall_model = metrics_df.loc[best_recall_idx, 'Модель']
best_recall_value = metrics_df.loc[best_recall_idx, 'Recall (Кукла)']

print(f"\n🏆 Лучшая модель по Recall (Кукла): {best_recall_model} ({best_recall_value})")


=== Шаг 3: Обучение моделей ===
Обучение Logistic Regression...
Accuracy: 0.7292 | Recall (Кукла): 0.6667
----------------------------------------
Обучение SVM (RBF kernel)...
Accuracy: 0.7708 | Recall (Кукла): 0.7500
----------------------------------------
Обучение Random Forest...
Accuracy: 0.6458 | Recall (Кукла): 0.5000
----------------------------------------

🏆 Лучшая модель по Recall (Кукла): SVM (RBF kernel) (0.7500)


In [48]:
# ==========================================
# ШАГ 4: СОХРАНЕНИЕ CSV
# ==========================================
print(f"\n=== Шаг 4: Сохранение результатов в {OUTPUT_CSV} ===")

csv_rows = []
for name, model_info in results.items():
    y_pred = model_info['y_pred']
    for i in range(len(y_test)):
        csv_rows.append({
            'Модель': name,
            'Файл': files_test[i],
            'Истинный класс': int(y_test[i]),
            'Предсказание': int(y_pred[i]),
            'Верно': 'Да' if y_pred[i] == y_test[i] else 'Нет',
            'Тип ошибки': 'OK' if y_pred[i] == y_test[i]
                         else ('FN (пропустил куклу)' if y_test[i] == 1 else 'FP (ложная тревога)'),
            'Аугментировано': 'Да' if aug_test[i] is not None else 'Нет'
        })

csv_df = pd.DataFrame(csv_rows)
csv_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"Сохранено {len(csv_df)} строк в {OUTPUT_CSV}")


=== Шаг 4: Сохранение результатов в results_augmented.csv ===
Сохранено 144 строк в results_augmented.csv


In [49]:
# ==========================================
# ШАГ 5: ГЕНЕРАЦИЯ HTML-ОТЧЁТА
# ==========================================
print(f"\n=== Шаг 5: Генерация HTML-отчёта {OUTPUT_HTML} ===")

def build_predictions_table_with_aug(model_name, y_pred_arr):
    rows = []
    tp = int(np.sum((y_pred_arr == 1) & (y_test == 1)))
    tn = int(np.sum((y_pred_arr == 0) & (y_test == 0)))
    fp = int(np.sum((y_pred_arr == 1) & (y_test == 0)))
    fn = int(np.sum((y_pred_arr == 0) & (y_test == 1)))
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    acc = (tp + tn) / len(y_test)

    stats_html = f'''
    <div class="model-stats">
        <b>Метрики:</b> Accuracy = {acc:.4f} | 
        Recall (Кукла) = {recall:.4f} | 
        Precision (Кукла) = {precision:.4f} |
        TP={tp}, TN={tn}, FP={fp}, FN={fn}
    </div>'''

    for i in range(len(y_test)):
        # Находим путь к оригиналу
        orig_filename = files_test[i].replace('aug_0_', '').replace('aug_1_', '')
        img_path = None
        for folder in [DATASET_FOLDER, ADD_POSITIVE_FOLDER, ADD_NEGATIVE_FOLDER]:
            test_path = os.path.join(folder, orig_filename)
            if os.path.exists(test_path):
                img_path = test_path
                break
        
        if img_path is None:
            continue
        
        img_b64 = img_to_base64(img_path)
        _, hog_img = get_hog_visualization(img_path)
        hog_b64 = array_to_base64(hog_img)
        
        # === АУГМЕНТАЦИЯ ДЛЯ ОТОБРАЖЕНИЯ (не влияет на обучение) ===
        img_orig = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
        img_aug_display = augment_image(img_rgb)
        aug_b64 = array_to_base64(img_aug_display)

        true_label = int(y_test[i])
        pred_label = int(y_pred_arr[i])
        is_correct = (pred_label == true_label)

        if true_label == 1 and pred_label == 0:
            err_type, row_style = 'FN', ' style="background:#ffebee;"'
        elif true_label == 0 and pred_label == 1:
            err_type, row_style = 'FP', ' style="background:#fff3e0;"'
        else:
            err_type, row_style = 'OK', ''

        rows.append(f'''
        <tr{row_style}>
            <td>{i+1}</td>
            <td>{files_test[i]}</td>
            <td>{true_label}</td>
            <td>{pred_label}</td>
            <td>{"✅" if is_correct else "❌"}</td>
            <td>{err_type}</td>
            <td><img src="data:image/jpeg;base64,{img_b64}" style="width:140px;height:140px;object-fit:cover;border-radius:4px;"></td>
            <td><img src="data:image/jpeg;base64,{aug_b64}" style="width:140px;height:140px;object-fit:cover;border-radius:4px;"></td>
            <td><img src="data:image/jpeg;base64,{hog_b64}" style="width:140px;height:140px;object-fit:cover;border-radius:4px;"></td>
        </tr>''')

    header = '''<table><thead><tr>
        <th>#</th><th>Файл</th><th>Истина</th><th>Предсказание</th>
        <th>Верно?</th><th>Тип</th><th>Оригинал</th><th>Аугментация</th><th>HOG</th>
    </tr></thead><tbody>'''

    return stats_html + header + ''.join(rows) + '</tbody></table>'

html = []

html.append(f'''<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Отчёт по классификации HOG (с аугментацией)</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 20px; background: #f5f5f5; color: #333; }}
        h1 {{ color: #2c3e50; }}
        h2 {{ color: #34495e; border-bottom: 2px solid #3498db; padding-bottom: 5px; }}
        h3 {{ color: #7f8c8d; }}
        table {{ border-collapse: collapse; margin: 20px 0; background: white; width: 100%; }}
        th, td {{ border: 1px solid #ddd; padding: 8px 12px; text-align: center; }}
        th {{ background: #3498db; color: white; }}
        tr:nth-child(even) {{ background: #f9f9f9; }}
        tr.highlight {{ background: #fff3cd !important; font-weight: bold; }}
        .gallery {{ display: flex; flex-wrap: wrap; gap: 15px; margin: 20px 0; }}
        .card {{ background: white; border: 1px solid #ddd; border-radius: 8px; padding: 10px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1); width: 460px; }}
        .card img {{ width: 100%; height: 200px; object-fit: cover; border-radius: 4px; }}
        .card .title {{ font-size: 11px; margin-top: 8px; word-wrap: break-word; }}
        .card .img-label {{ font-size: 10px; color: #7f8c8d; margin: 4px 0 2px 0; font-weight: bold; }}
        .error {{ border-left: 4px solid #e74c3c; }}
        .correct {{ border-left: 4px solid #27ae60; }}
        .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px;
                  font-size: 10px; font-weight: bold; color: white; margin-top: 4px; }}
        .badge-ok {{ background: #27ae60; }}
        .badge-fn {{ background: #e74c3c; }}
        .badge-fp {{ background: #f39c12; }}
        .section {{ background: white; padding: 20px; margin: 20px 0; border-radius: 8px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .trophy {{ background: linear-gradient(135deg, #f1c40f 0%, #f39c12 100%); color: white;
                   padding: 15px 25px; border-radius: 8px; margin: 20px 0; font-size: 18px; }}
        .img-row {{ display: flex; gap: 8px; }}
        .img-col {{ flex: 1; }}
        .model-selector {{ margin: 15px 0; padding: 12px; background: #ecf0f1; border-radius: 6px; }}
        .model-selector label {{ font-weight: bold; margin-right: 10px; }}
        .model-selector select {{ padding: 8px 12px; font-size: 14px; border: 1px solid #bdc3c7; border-radius: 4px; cursor: pointer; }}
        .model-block {{ display: none; }}
        .model-block.active {{ display: block; }}
        .model-stats {{ background: #f8f9fa; padding: 12px; border-radius: 6px; margin: 10px 0; font-size: 14px; }}
        .table-wrapper {{ overflow-x: auto; }}
    </style>
</head>
<body>
    <h1>📊 Отчёт по классификации изображений (HOG + Аугментация)</h1>
    <p><b>Дата:</b> {datetime.now().strftime("%d.%m.%Y %H:%M")}</p>
    <p><b>Параметры HOG:</b> image_size={IMAGE_SIZE}, pixels_per_cell={PIXELS_PER_CELL},
       cells_per_block={CELLS_PER_BLOCK}, orientations={ORIENTATIONS}</p>
    <p><b>Всего изображений:</b> {len(filenames)} (включая аугментированные) |
       <b>Обучение:</b> {len(X_train)} | <b>Тест:</b> {len(X_test)}</p>

    <div class="trophy">
        🏆 Лучшая модель по Recall (Кукла): <b>{best_recall_model}</b> — Recall = {best_recall_value}
    </div>
''')

# Сводная таблица
html.append('<div class="section"><h2>📈 Сводная таблица моделей</h2>')
html.append('<table><thead><tr>')
for col in metrics_df.columns:
    html.append(f'<th>{col}</th>')
html.append('</tr></thead><tbody>')
for idx, row in metrics_df.iterrows():
    cls = ' class="highlight"' if idx == best_recall_idx else ''
    html.append(f'<tr{cls}>')
    for col in metrics_df.columns:
        html.append(f'<td>{row[col]}</td>')
    html.append('</tr>')
html.append('</tbody></table>')
html.append('<p><i>🟡 Жёлтым выделена лучшая модель по Recall (Кукла)</i></p>')
html.append('</div>')




# По каждой модели
for name, model_info in results.items():
    y_pred = model_info['y_pred']
    acc = model_info['accuracy']
    
    html.append(f'<div class="section">')
    html.append(f'<h2>🤖 Модель: {name} (Accuracy: {acc:.4f})</h2>')
    
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Не кукла (0)', 'Кукла (1)'])
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title(f"Матрица ошибок: {name}")
    html.append(f'<h3>Матрица ошибок</h3>')
    html.append(f'<img src="data:image/png;base64,{fig_to_base64(fig)}" style="max-width:400px;">')
    
    # Галерея ошибок (только оригинал + HOG, без аугментации)
    error_indices = np.where(y_pred != y_test)[0]
    html.append(f'<h3>Ошибки ({len(error_indices)} шт.) — оригинал + HOG</h3>')

    if len(error_indices) > 0:
        html.append('<div class="gallery">')
        for idx in error_indices[:6]:
            orig_filename = files_test[idx].replace('aug_0_', '').replace('aug_1_', '')
            img_path = None
            for folder in [DATASET_FOLDER, ADD_POSITIVE_FOLDER, ADD_NEGATIVE_FOLDER]:
                test_path = os.path.join(folder, orig_filename)
                if os.path.exists(test_path):
                    img_path = test_path
                    break
            
            if img_path is None:
                continue
            
            img_b64 = img_to_base64(img_path)
            _, hog_img = get_hog_visualization(img_path)
            hog_b64 = array_to_base64(hog_img)
            
            true_label = int(y_test[idx])
            pred_label = int(y_pred[idx])
            
            if true_label == 1 and pred_label == 0:
                badge_class = 'badge-fn'
                badge_text = 'FN: пропустил куклу'
            else:
                badge_class = 'badge-fp'
                badge_text = 'FP: ложная тревога'
            
            html.append(f'''
            <div class="card error">
                <div class="img-row">
                    <div class="img-col">
                        <div class="img-label">Оригинал</div>
                        <img src="data:image/jpeg;base64,{img_b64}" style="width:200px;height:200px;object-fit:cover;">
                    </div>
                    <div class="img-col">
                        <div class="img-label">HOG</div>
                        <img src="data:image/jpeg;base64,{hog_b64}" style="width:200px;height:200px;object-fit:cover;">
                    </div>
                </div>
                <div class="title">
                    <b>{files_test[idx]}</b><br>
                    Истина: {true_label} | Предсказание: {pred_label}<br>
                    <span class="badge {badge_class}">{badge_text}</span>
                </div>
            </div>''')
        html.append('</div>')
    else:
        html.append('<p>✅ Ошибок не обнаружено!</p>')
    
    html.append('</div>')
# ==========================================
# СЕКЦИЯ: ЗАГРУЗКА ИЗОБРАЖЕНИЙ
# ==========================================
html.append('''
<div class="section">
    <h2>🧪 Тест на своей картинке</h2>
    <p>Загрузите изображение — все три модели сделают предсказание прямо в браузере.</p>
    <div class="upload-area">
        <input type="file" id="fileInput" accept="image/*">
        <button onclick="uploadImage()">Предсказать</button>
    </div>
    <div id="uploadResult" style="margin-top:20px;"></div>
</div>
''')
# Интерактивная секция с выбором модели
html.append('<div class="section">')
html.append('<h2>🔍 Все предсказания на тестовой выборке</h2>')

options_html = ''
for idx, name in enumerate(results.keys()):
    selected = ' selected' if idx == 0 else ''
    options_html += f'<option value="model_{idx}"{selected}>{name}</option>'

html.append(f'''
<div class="model-selector">
    <label for="modelSelect">Выберите модель:</label>
    <select id="modelSelect" onchange="switchModel()">
        {options_html}
    </select>
</div>
''')

for idx, (name, model_info) in enumerate(results.items()):
    active = ' active' if idx == 0 else ''
    html.append(f'<div id="model_{idx}" class="model-block{active}">')
    html.append(f'<h3>🤖 {name}</h3>')
    html.append(build_predictions_table_with_aug(name, model_info['y_pred']))
    html.append('</div>')

html.append('</div>')

html.append('''
<script>
function switchModel() {
    const sel = document.getElementById('modelSelect');
    const chosen = sel.value;
    document.querySelectorAll('.model-block').forEach(div => {
        div.classList.remove('active');
    });
    document.getElementById(chosen).classList.add('active');
}
</script>
</body></html>''')
html.append('''
<script>
let previewDataUrl = null;

function uploadImage() {
    const input = document.getElementById('fileInput');
    const resultDiv = document.getElementById('uploadResult');
    
    if (!input.files || !input.files[0]) {
        alert('Сначала выберите файл!');
        return;
    }
    
    const file = input.files[0];
    const formData = new FormData();
    formData.append('file', file);
    
    const reader = new FileReader();
    reader.onload = function(e) {
        previewDataUrl = e.target.result;
        resultDiv.innerHTML = `
            <img src="${previewDataUrl}" class="preview-img">
            <p>⏳ Отправка на сервер...</p>
        `;
    };
    reader.readAsDataURL(file);
    
    fetch('http://localhost:5000/predict', {
        method: 'POST',
        body: formData
    })
    .then(response => response.json())
    .then(data => {
        if (data.error) {
            resultDiv.innerHTML = `<p style="color:red;">Ошибка: ${data.error}</p>`;
            return;
        }
        
        let html = `
            <div class="img-row" style="max-width:700px;">
                <div class="img-col">
                    <div class="img-label">Оригинал</div>
                    <img src="${previewDataUrl}" class="preview-img" style="width:100%;max-width:320px;height:240px;object-fit:cover;border-radius:6px;">
                </div>
                <div class="img-col">
                    <div class="img-label">HOG-визуализация</div>
                    <img src="data:image/jpeg;base64,${data.hog_image}" class="preview-img" style="width:100%;max-width:320px;height:240px;object-fit:cover;border-radius:6px;">
                </div>
            </div>
            <h3>Результаты предсказания:</h3>
        `;
        
        for (const [modelName, pred] of Object.entries(data.predictions)) {
            const isDoll = pred.class === 1;
            const cardClass = isDoll ? 'doll' : 'not-doll';
            const label = isDoll ? '🐱 КУКЛА' : '❌ НЕ КУКЛА';
            const probText = pred.probability !== null ? 
                ` (уверенность: ${(pred.probability * 100).toFixed(1)}%)` : '';
            
            html += `
                <div class="prediction-card ${cardClass}">
                    <b>${modelName}:</b> ${label}${probText}
                </div>
            `;
        }
        resultDiv.innerHTML = html;
    })
    .catch(err => {
        resultDiv.innerHTML = `<p style="color:red;">Ошибка соединения с сервером: ${err.message}<br>Убедитесь, что app.py запущен</p>`;
    });
}
</script>
''')
with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write('\n'.join(html))

print(f"✅ HTML-отчёт сохранён: {OUTPUT_HTML}")
print(f"📄 CSV-файл сохранён: {OUTPUT_CSV}")
print(f"\nОткройте {OUTPUT_HTML} в браузере для просмотра отчёта!")


=== Шаг 5: Генерация HTML-отчёта report_augmented.html ===
✅ HTML-отчёт сохранён: report_augmented.html
📄 CSV-файл сохранён: results_augmented.csv

Откройте report_augmented.html в браузере для просмотра отчёта!


In [50]:
import pickle

models_data = {}
for name, model in models.items():
    if hasattr(model, 'named_steps'):
        scaler = model.named_steps['standardscaler']
        clf = model.named_steps[list(model.named_steps.keys())[-1]]
        models_data[name] = {'model': clf, 'scaler': scaler}
    else:
        models_data[name] = {'model': model, 'scaler': None}

with open('models.pkl', 'wb') as f:
    pickle.dump(models_data, f)
print("✅ Модели сохранены в models.pkl")

✅ Модели сохранены в models.pkl
